# Cluster Summary

**Fine-tuned EmbeddingGemma 2 version** of `main/code/02_cluster_summary.ipynb`: the DoubleMLDeep product embeddings are replaced by [EmbeddingGemma 2](https://huggingface.co/google/embeddinggemma-2) embeddings fine-tuned with LoRA and the loss of the DoubleMLDeep model on the training sample (`finetune_gemma.py`, `00_finetune_gemma_colab.ipynb`; text + image, Matryoshka 256 dimensions). Datasets are read from `Gemma-tuned/data`, raw data from `main/data`, outputs are written to `Gemma-tuned/outputs`.

Requires the outputs of `04_evaluation.ipynb` (`sorted_effects.csv`); the cluster labels are the short versions of the headlines in `02_cluster_centroid_products.ipynb`.

In [1]:
import numpy as np
import pandas as pd

df_val = pd.read_csv("../data/dataset_txt_only_False_embeddings_True_val.zip")
#df_train = pd.read_csv("../data/dataset_txt_only_False_embeddings_True_train.zip")
df_effects = pd.read_csv("../outputs/04_evaluation/sorted_effects.csv")

df_full = df_val # pd.concat([df_val, df_train], ignore_index=True)
df_full.head()

,ASIN,date,index,Q_t,PRICE,P_bb_t,text,window,REVIEW_COUNT,RATING,...,Delta_Q_t,Delta_P_bb_t,pred_ml_l,pred_ml_m,pred_ml_l_diff,pred_ml_m_diff,pred_ml_l_lag_1,pred_ml_m_lag_1,pred_ml_l_diff_lag_1,pred_ml_m_diff_lag_1
0,1947494961,2023-01-30,1,-10.685092,3.068750,3.318075,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1501.214286,4.700000,...,NaN,NaN,-10.608895,2.811658,NaN,NaN,NaN,NaN,NaN,NaN
1,1947494961,2023-02-27,9,-10.728496,3.149694,3.435898,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1552.535714,4.700000,...,-0.043404,0.117823,-10.608895,2.811658,-0.060465,-0.003964,-10.849645,2.824807,-0.032787,-0.013729
2,1947494961,2023-03-27,17,-11.041387,3.201483,3.457206,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1567.000000,4.700000,...,-0.312891,0.021309,-10.608895,2.811658,0.000150,-0.002530,-10.825026,2.825181,0.138726,-0.013659
3,1947494961,2023-04-24,25,-11.055897,3.357830,3.425530,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1517.714286,4.666667,...,-0.014510,-0.031676,-10.608895,2.811658,-0.011212,-0.002330,-10.818141,2.825185,0.028958,-0.008854
4,1947494961,2023-05-22,33,-11.157410,3.330417,3.375880,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1580.500000,4.600000,...,-0.101513,-0.049650,-10.608895,2.811658,-0.031883,-0.002634,-10.831144,2.824067,0.021158,-0.010682


In [2]:
df_effects

,ASIN,date,SortEff_lin,SortEff_tab_dml,SortEff_deep_dml
0,1947494961,2023-02-27,-0.714728,-0.745703,-0.718053
1,1947494961,2023-03-27,-0.704648,-0.738301,-0.714290
2,1947494961,2023-04-24,-0.662543,-0.692221,-0.655572
3,1947494961,2023-05-22,-0.661796,-0.690371,-0.651603
4,1947494961,2023-06-19,-0.650223,-0.675918,-0.630395
...,...,...,...,...,...
38036,B0CPSR7XM1,2024-01-29,-0.535817,-0.532740,-0.571373
38037,B0CQ1SLW1P,2024-01-29,-0.585271,-0.598178,-0.600067
38038,B0CQ1XKDQX,2024-01-29,-0.609705,-0.623177,-0.615195
38039,B0CQWJ3247,2024-01-29,-0.762456,-0.811155,-0.884288


In [3]:
df_val.set_index(["ASIN", "date"]).join(df_effects.set_index(["ASIN", "date"]), how="left")

index        Q_t     PRICE    P_bb_t  \
ASIN       date                                                
1947494961 2023-01-30       1 -10.685092  3.068750  3.318075   
           2023-02-27       9 -10.728496  3.149694  3.435898   
           2023-03-27      17 -11.041387  3.201483  3.457206   
           2023-04-24      25 -11.055897  3.357830  3.425530   
           2023-05-22      33 -11.157410  3.330417  3.375880   
...                       ...        ...       ...       ...   
B0CQWJ3247 2024-01-01  592585  -9.410761  2.078191  2.078191   
           2024-01-29  592593  -9.367674  2.116772  2.116772   
B0CQWJZGSY 2024-01-01  592741  -9.376208  1.885553  1.885553   
           2024-01-29  592749  -9.461319  1.998193  1.998193   
B0CRDLRP38 2024-01-29  592905 -11.850726  3.295466  3.295466   

                                                                    text  \
ASIN       date                                                            
1947494961 2023-01-30  Gale Force Nine - Aliens: Another Glorious Day...   
           2023-02-27  Gale Force Nine - Aliens: Another Glorious Day...   
           2023-03-27  Gale Force Nine - Aliens: Another Glorious Day...   
           2023-04-24  Gale Force Nine - Aliens: Another Glorious Day...   
           2023-05-22  Gale Force Nine - Aliens: Another Glorious Day...   
...                                                                  ...   
B0CQWJ3247 2024-01-01  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
           2024-01-29  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
B0CQWJZGSY 2024-01-01  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
           2024-01-29  Car Toys Diecast 1:55 Metal Model Vehical Toy ...   
B0CRDLRP38 2024-01-29  Keejoy 2-in-1 Dinosaur Truck Toy with 6 Stylis...   

                       window  REVIEW_COUNT    RATING  \
ASIN       date                                         
1947494961 2023-01-30    28.0   1501.214286  4.700000   
           2023-02-27    28.0   1552.535714  4.700000   
           2023-03-27    28.0   1567.000000  4.700000   
           2023-04-24    28.0   1517.714286  4.666667   
           2023-05-22    28.0   1580.500000  4.600000   
...                       ...           ...       ...   
B0CQWJ3247 2024-01-01    28.0      3.000000  3.500000   
           2024-01-29    28.0      2.357143  3.982143   
B0CQWJZGSY 2024-01-01    28.0      3.000000  3.500000   
           2024-01-29    28.0      2.357143  3.982143   
B0CRDLRP38 2024-01-29    28.0     11.166667  3.708696   

                       New Offer Count: Current  \
ASIN       date                                   
1947494961 2023-01-30                         2   
           2023-02-27                         2   
           2023-03-27                         2   
           2023-04-24                         2   
           2023-05-22                         2   
...                                         ...   
B0CQWJ3247 2024-01-01                         1   
           2024-01-29                         1   
B0CQWJZGSY 2024-01-01                         1   
           2024-01-29                         1   
B0CRDLRP38 2024-01-29                         1   

                       Count of retrieved live offers: New, FBA  ...  \
ASIN       date                                                  ...   
1947494961 2023-01-30                                         1  ...   
           2023-02-27                                         1  ...   
           2023-03-27                                         1  ...   
           2023-04-24                                         1  ...   
           2023-05-22                                         1  ...   
...                                                         ...  ...   
B0CQWJ3247 2024-01-01                                         0  ...   
           2024-01-29                                         0  ...   
B0CQWJZGSY 2024-01-01                                         0  ...   
    

In [4]:
len(df_val.set_index(["ASIN", "date"]).join(df_effects.set_index(["ASIN", "date"]), how="left").dropna(subset=["SortEff_deep_dml"]))

38041

In [5]:
sim_cols = [
 'similarity_cluster_0',
 'similarity_cluster_1',
 'similarity_cluster_2',
 'similarity_cluster_3',
 'similarity_cluster_4',
]

In [6]:
np.exp(10.685092)

np.float64(43699.50264947328)

In [7]:
# assign cluster label for each row based on the highest similarity score in sim_cols

df_full['cluster_label'] = df_full[sim_cols].idxmax(axis=1)
df_full['cluster_label'] = df_full['cluster_label'].str.replace('similarity_cluster_', '').astype(int)
df_full['SR'] = np.exp(-df_full["Q_t"])
df_full['Price'] = np.exp(df_full["P_bb_t"])
df_full.head()


,ASIN,date,index,Q_t,PRICE,P_bb_t,text,window,REVIEW_COUNT,RATING,...,pred_ml_m,pred_ml_l_diff,pred_ml_m_diff,pred_ml_l_lag_1,pred_ml_m_lag_1,pred_ml_l_diff_lag_1,pred_ml_m_diff_lag_1,cluster_label,SR,Price
0,1947494961,2023-01-30,1,-10.685092,3.068750,3.318075,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1501.214286,4.700000,...,2.811658,NaN,NaN,NaN,NaN,NaN,NaN,0,43699.500000,27.607143
1,1947494961,2023-02-27,9,-10.728496,3.149694,3.435898,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1552.535714,4.700000,...,2.811658,-0.060465,-0.003964,-10.849645,2.824807,-0.032787,-0.013729,0,45638.000000,31.059286
2,1947494961,2023-03-27,17,-11.041387,3.201483,3.457206,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1567.000000,4.700000,...,2.811658,0.000150,-0.002530,-10.825026,2.825181,0.138726,-0.013659,0,62404.142857,31.728214
3,1947494961,2023-04-24,25,-11.055897,3.357830,3.425530,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1517.714286,4.666667,...,2.811658,-0.011212,-0.002330,-10.818141,2.825185,0.028958,-0.008854,0,63316.214286,30.738929
4,1947494961,2023-05-22,33,-11.157410,3.330417,3.375880,Gale Force Nine - Aliens: Another Glorious Day...,28.0,1580.500000,4.600000,...,2.811658,-0.031883,-0.002634,-10.831144,2.824067,0.021158,-0.010682,0,70081.214286,29.250000


In [8]:
import os
import numpy as np


# Short labels from the LLM headline exercise (used as readable cluster names)
cluster_headlines_short = {
    0: "Fingerboard Parts & Accessories",
    1: "Licensed Character Track Playsets",
    2: "Toddler Carrier Truck Sets",
    3: "1:64 Collectible Die-Cast",
    4: "Alloy Pull-Back Model Cars",
}


# ── Load per-observation deep-DML elasticity and merge on (ASIN, date) ──────
sorted_eff = pd.read_csv("../outputs/04_evaluation/sorted_effects.csv")
sorted_eff["date"] = sorted_eff["date"].astype(str)
df_full["date"] = df_full["date"].astype(str)

df_char = df_full.merge(
    sorted_eff[["ASIN", "date", "SortEff_deep_dml"]],
    on=["ASIN", "date"],
    how="left",
).dropna(subset=["SortEff_deep_dml"])  # keep only rows with elasticity estimates

# ── Build per-cluster statistics ─────────────────────────────────────────────
rows = []
for c in range(5):
    sub = df_char[df_char["cluster_label"] == c]
    sub_asin = sub.groupby("ASIN").first().reset_index()  # one row per product

    n_prod         = int(sub["ASIN"].nunique())
    n_obs          = len(sub)
    mean_price     = np.exp(sub["P_bb_t"]).mean()
    median_SR      = int(np.exp(-sub_asin["Q_t"]).median())
    mean_rating    = sub["RATING"].mean()
    # Product-level median review count (avoids repeating the same product
    # across many time periods inflating the statistic)
    median_reviews = int(sub_asin["REVIEW_COUNT"].median())

    # Top-2 subcategories by number of distinct products
    top_cats = (
        sub_asin["subcat_aggregated"]
        .value_counts()
        .head(2)
        .index.tolist()
    )
    cat_str = " / ".join(top_cats)

    # Mean elasticity (observations matched to sorted_effects only)
    mean_elas = sub["SortEff_deep_dml"].mean()

    rows.append(
        {
            "Cluster": c,
            "Label": cluster_headlines_short[c],
            "# Products": n_prod,
            "# Observations": n_obs,
            "Mean Price ($)": round(mean_price, 2),
            "Median Sales Rank": median_SR,
            "Mean Rating": round(mean_rating, 2),
            "Median Reviews": median_reviews,
            "Dominant Product Type": cat_str,
            "Mean Elasticity": round(mean_elas, 3),
        }
    )

df_summary = pd.DataFrame(rows).set_index("Cluster")

# ── Save ─────────────────────────────────────────────────────────────────────
os.makedirs("../outputs/R2", exist_ok=True)
df_summary.to_csv(os.path.join("../outputs/R2", "cluster_summary_table.csv"))

# ── Display ──────────────────────────────────────────────────────────────────
df_summary.style.format(
    {
        "Mean Price ($)": "${:.2f}",
        "Mean Rating": "{:.2f}",
        "Mean Elasticity": "{:.3f}",
    }
)

,Label,# Products,# Observations,Mean Price ($),Median Sales Rank,Mean Rating,Median Reviews,Dominant Product Type,Mean Elasticity
Cluster,,,,,,,,,
0,Fingerboard Parts & Accessories,552,5976,$15.15,101650,4.42,50,Skateboards / Train Sets,-0.507
1,Licensed Character Track Playsets,771,8984,$44.41,71219,4.58,408,Trucks / Residual,-0.810
2,Toddler Carrier Truck Sets,891,8327,$27.00,64852,4.36,72,Trucks / Train Sets,-0.635
3,1:64 Collectible Die-Cast,824,8184,$14.68,89867,4.58,19,Cars & Race Cars / Trucks,-0.723
4,Alloy Pull-Back Model Cars,670,6570,$22.97,90474,4.48,23,Cars & Race Cars / Residual,-0.622
